In [ ]:
# Install dependencies
!pip install diffusers==0.31.0 transformers==4.38.2 peft==0.9.0 accelerate==0.27.2 -q
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
# Imports
import torch
import torch.nn.functional as F
import pandas as pd
import os
from google.colab import userdata
from huggingface_hub import login
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    DDPMScheduler,
    AutoencoderKL,
    UNet2DConditionModel
)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
# Hugging Face login
hf_token = userdata.get('HF_TOKEN')
login(hf_token)
# Dataset class
class PennCampusDataset(Dataset):
    def __init__(self, csv_path, image_dir, edge_dir, tokenizer, resolution=512):
        self.df = pd.read_csv(csv_path)
        self.df = self._validate_files(self.df, image_dir, edge_dir)

        self.image_dir = image_dir
        self.edge_dir = edge_dir
        self.tokenizer = tokenizer
        self.resolution = resolution

        # Image transform: normalize to [-1, 1]
        self.image_transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

        # Edge transform: normalize to [0, 1]
        self.edge_transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
        ])

        print(f"Dataset initialized with {len(self.df)} valid samples")

    def _validate_files(self, df, image_dir, edge_dir):
        valid_rows = []
        missing_count = 0

        for idx, row in df.iterrows():
            filename = row['filename']
            image_path = os.path.join(image_dir, filename)
            edge_path = os.path.join(edge_dir, filename)

            if os.path.exists(image_path) and os.path.exists(edge_path):
                valid_rows.append(row)
            else:
                missing_count += 1
                print(f"Skipping missing file: {filename}")

        if missing_count > 0:
            print(f"Warning: {missing_count} files missing and skipped")

        return pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        prompt = row['prompt']

        # Load image
        image_path = os.path.join(self.image_dir, filename)
        image = Image.open(image_path).convert("RGB")

        # Load edge map
        edge_path = os.path.join(self.edge_dir, filename)
        edge = Image.open(edge_path).convert("RGB")

        # Apply transforms
        pixel_values = self.image_transform(image)
        conditioning_pixel_values = self.edge_transform(edge)

        # Tokenize prompt
        tokenized = self.tokenizer(
            prompt,
            max_length=self.tokenizer.model_max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "pixel_values": pixel_values,
            "conditioning_pixel_values": conditioning_pixel_values,
            "input_ids": tokenized.input_ids.squeeze(0),
            "prompt": prompt,
        }


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "conditioning_pixel_values": torch.stack([b["conditioning_pixel_values"] for b in batch]),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
    }
# Validation function - computes numerical loss on validation set
def compute_validation_loss(unet, controlnet, val_loader, noise_scheduler, vae, text_encoder, device, dtype):
    """Compute validation loss on held-out dataset."""
    unet.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device, dtype=dtype)
            conditioning_pixel_values = batch["conditioning_pixel_values"].to(device, dtype=dtype)
            input_ids = batch["input_ids"].to(device)

            # Encode to latents
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            # Add noise
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Get text embeddings
            encoder_hidden_states = text_encoder(input_ids)[0]

            # ControlNet forward
            down_block_res_samples, mid_block_res_sample = controlnet(
                noisy_latents, timesteps, encoder_hidden_states,
                controlnet_cond=conditioning_pixel_values,
                return_dict=False,
            )

            # UNet forward
            noise_pred = unet(
                noisy_latents, timesteps, encoder_hidden_states,
                down_block_additional_residuals=down_block_res_samples,
                mid_block_additional_residual=mid_block_res_sample,
            ).sample

            # Compute loss
            loss = F.mse_loss(noise_pred.float(), noise.float())
            total_loss += loss.item()
            num_batches += 1

    avg_val_loss = total_loss / num_batches
    unet.train()
    return avg_val_loss


# Optional: Visual validation - generates sample images
VISUAL_VALIDATION_SAMPLES = [
    {
        "edge_map": "/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap/vp_sunny_day_front_facing.jpg",
        "prompt": "PENNVP, Van Pelt library, modernist brick and concrete facade, large rectangular windows, flat roofline, five-story library building, front facing view, night time, dark clear sky, warm lamppost glow, artificial lighting, illuminated windows, lit facade, Penn campus, University of Pennsylvania, Philadelphia, architectural photography, Upenn",
        "name": "vp_sunny_to_night"
    },
    {
        "edge_map": "/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap/collegehall_sunny_day_front_facing_right.jpg",
        "prompt": "PENNCH, College Hall, gothic collegiate architecture, green serpentine stone facade, rainy day, wet pavement, puddle reflections, overcast grey sky, Penn campus, University of Pennsylvania, Philadelphia, architectural photography, Upenn",
        "name": "ch_sunny_to_rainy"
    },
]

def generate_visual_validation(unet, controlnet, epoch, output_dir):
    """Generate visual validation samples."""
    print(f"  Generating visual samples...")

    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "/content/drive/MyDrive/CIS_5190_Project_Material/Models/sd-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.bfloat16,
        local_files_only=True
    ).to("cuda")

    pipe.unet = unet
    pipe.set_progress_bar_config(disable=True)

    unet.eval()

    for sample in VISUAL_VALIDATION_SAMPLES:
        edge_map = Image.open(sample["edge_map"]).convert("RGB").resize((512, 512))

        generated = pipe(
            prompt=sample["prompt"],
            image=edge_map,
            num_inference_steps=30,
            guidance_scale=7.5,
            controlnet_conditioning_scale=0.8,
        ).images[0]

        output_path = os.path.join(output_dir, f"epoch_{epoch}_{sample['name']}.png")
        generated.save(output_path)

    unet.train()
    del pipe
    torch.cuda.empty_cache()
# Training configuration
MODEL_PATH = "/content/drive/MyDrive/CIS_5190_Project_Material/Models/sd-v1-5"
CONTROLNET_PATH = "/content/drive/MyDrive/CIS_5190_Project_Material/Models/controlnet-canny"
OUTPUT_DIR = "/content/drive/MyDrive/CIS_5190_Project_Material/LoRA_Checkpoints"
VALIDATION_DIR = "/content/drive/MyDrive/CIS_5190_Project_Material/Validation_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)

# Hyperparameters
RESOLUTION = 512
BATCH_SIZE = 4  # Reduce to 2 if OOM
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
SAVE_EVERY_N_EPOCH = 5
VALIDATE_EVERY_N_EPOCHS = 5
LORA_RANK = 8
LORA_ALPHA = 32  # Higher influence
MAX_GRAD_NORM = 1.0

device = "cuda"
dtype = torch.bfloat16  # Consistent bfloat16 everywhere

print("Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA} (scaling: {LORA_ALPHA/LORA_RANK})")
print(f"  Device: {device}")
print(f"  Dtype: {dtype}")
# Load model components
print("Loading model components...")

tokenizer = CLIPTokenizer.from_pretrained(MODEL_PATH, subfolder="tokenizer", local_files_only=True)
text_encoder = CLIPTextModel.from_pretrained(
    MODEL_PATH, subfolder="text_encoder", torch_dtype=dtype, local_files_only=True
).to(device)
vae = AutoencoderKL.from_pretrained(
    MODEL_PATH, subfolder="vae", torch_dtype=dtype, local_files_only=True
).to(device)
unet = UNet2DConditionModel.from_pretrained(
    MODEL_PATH, subfolder="unet", torch_dtype=dtype, local_files_only=True
).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_PATH, subfolder="scheduler", local_files_only=True)
controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_PATH, torch_dtype=dtype, local_files_only=True
).to(device)

# Freeze components
text_encoder.requires_grad_(False)
vae.requires_grad_(False)
unet.requires_grad_(False)
controlnet.requires_grad_(False)

print("✓ Models loaded")
# Inject LoRA into UNet
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    lora_dropout=0.1,
    bias="none",
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()
unet.enable_gradient_checkpointing()

print("✓ LoRA injected into UNet")
# Create datasets and dataloaders
# NOTE: Run the create_train_val_split.py script first to generate these CSVs!

print("Loading datasets...")

train_dataset = PennCampusDataset(
    csv_path="/content/drive/MyDrive/CIS_5190_Project_Material/penn_lora_train.csv",
    image_dir="/content/drive/MyDrive/CIS_5190_Project_Material/Images",
    edge_dir="/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap",
    tokenizer=tokenizer,
    resolution=RESOLUTION,
)

val_dataset = PennCampusDataset(
    csv_path="/content/drive/MyDrive/CIS_5190_Project_Material/penn_lora_val.csv",
    image_dir="/content/drive/MyDrive/CIS_5190_Project_Material/Images",
    edge_dir="/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap",
    tokenizer=tokenizer,
    resolution=RESOLUTION,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
)

print(f"\n{'='*70}")
print(f"Dataset ready:")
print(f"  Training: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"  Validation: {len(val_dataset)} images ({len(val_loader)} batches)")
print(f"  Total training steps: {len(train_loader) * NUM_EPOCHS}")
print(f"{'='*70}")
# Optimizer
trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

print(f"✓ Optimizer created with {len(trainable_params)} parameter groups")
# Training loop
print(f"\n{'='*70}")
print("Starting training...")
print(f"{'='*70}\n")

# Enable A100 optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

unet.train()
controlnet.eval()
global_step = 0
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for step, batch in enumerate(progress_bar):
        # Load data
        pixel_values = batch["pixel_values"].to(device, dtype=dtype, non_blocking=True)
        conditioning_pixel_values = batch["conditioning_pixel_values"].to(device, dtype=dtype, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)

        # Encode image to latent space
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

        # Sample noise and timesteps
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Encode text prompt
        with torch.no_grad():
            encoder_hidden_states = text_encoder(input_ids)[0]

        # Forward pass with ControlNet
        with torch.autocast("cuda", dtype=torch.bfloat16):
            # ControlNet features from edge maps
            down_block_res_samples, mid_block_res_sample = controlnet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=conditioning_pixel_values,
                return_dict=False,
            )

            # UNet prediction with ControlNet conditioning
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states,
                down_block_additional_residuals=down_block_res_samples,
                mid_block_additional_residual=mid_block_res_sample,
            ).sample

            # MSE loss
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")

        # Backprop
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        # Track loss
        true_loss = loss.item()
        epoch_loss += true_loss
        progress_bar.set_postfix({"loss": f"{true_loss:.4f}"})

    avg_train_loss = epoch_loss / len(train_loader)

    # Validation
    if (epoch + 1) % VALIDATE_EVERY_N_EPOCHS == 0:
        val_loss = compute_validation_loss(
            unet, controlnet, val_loader, noise_scheduler,
            vae, text_encoder, device, dtype
        )

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")

        # Track best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"  ✓ New best validation loss!")

        # Generate visual samples
        generate_visual_validation(unet, controlnet, epoch + 1, VALIDATION_DIR)
    else:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Loss = {avg_train_loss:.4f}")

    # Save checkpoint
    if (epoch + 1) % SAVE_EVERY_N_EPOCH == 0:
        save_path = os.path.join(OUTPUT_DIR, f"lora_epoch_{epoch+1}")
        unet.save_pretrained(save_path)
        print(f"  ✓ Checkpoint saved: {save_path}")

# Final save
final_path = os.path.join(OUTPUT_DIR, "lora_final")
unet.save_pretrained(final_path)

print(f"\n{'='*70}")
print(f"Training complete!")
print(f"  Final model: {final_path}")
print(f"  Best validation loss: {best_val_loss:.4f}")
print(f"{'='*70}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch version:

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

✓ Models loaded
trainable params: 1,594,368 || all params: 861,115,332 || trainable%: 0.1852
✓ LoRA injected into UNet
Loading datasets...
Dataset initialized with 95 valid samples
Dataset initialized with 16 valid samples

Dataset ready:
  Training: 95 images (24 batches)
  Validation: 16 images (4 batches)
  Total training steps: 480
✓ Optimizer created with 256 parameter groups

Starting training...



Epoch 1/20:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 1/20: Train Loss = 0.1185


...


Epoch 20/20:
  Train Loss: 0.1395
  Val Loss:   0.0995
  Generating visual samples...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  ✓ Checkpoint saved: /content/drive/MyDrive/CIS_5190_Project_Material/LoRA_Checkpoints/lora_epoch_20

Training complete!
  Final model: /content/drive/MyDrive/CIS_5190_Project_Material/LoRA_Checkpoints/lora_final
  Best validation loss: 0.0509
